# unzipp miniDL subset

In [ ]:
from pathlib import Path
import zipfile, tarfile, shutil, hashlib

def human_bytes(n):
    for unit in ["B","KB","MB","GB","TB"]:
        if n < 1024:
            return f"{n:.2f} {unit}"
        n /= 1024
    return f"{n:.2f} PB"

def file_head(path: Path, n=16):
    try:
        with path.open("rb") as f:
            return f.read(n)
    except Exception:
        return b""

def md5sum(path: Path, block=1<<20):
    h = hashlib.md5()
    with path.open("rb") as f:
        while True:
            b = f.read(block)
            if not b: break
            h.update(b)
    return h.hexdigest()

def guess_from_magic(head: bytes):
    # Common signatures
    if head.startswith(b"PK\x03\x04") or head.startswith(b"PK\x05\x06") or head.startswith(b"PK\x07\x08"):
        return "zip"
    if head.startswith(b"\x1f\x8b\x08"):
        return "gzip (likely .tar.gz)"
    if head.startswith(b"BZh"):
        return "bzip2 (likely .tar.bz2)"
    if head.startswith(b"\xfd7zXZ"):
        return "xz (likely .tar.xz)"
    if head.startswith(b"ustar") or b"ustar" in head:
        return "tar (ustar)"
    if head[:5].lower().startswith(b"<?xml") or head[:5].lower().startswith(b"<!doc") or head[:4].lower()==b"<htm":
        return "html (probably a download/login page saved as .zip)"
    return "unknown"

def smart_extract(archive_path: Path, out_dir: Path = None):
    if out_dir is None:
        out_dir = archive_path.parent

    if not archive_path.exists():
        raise FileNotFoundError(f"Archive not found: {archive_path}")

    size = archive_path.stat().st_size
    head = file_head(archive_path, 64)
    kind = guess_from_magic(head)
    print(f"[INFO] File: {archive_path.name} | Size: {human_bytes(size)} | MD5: {md5sum(archive_path)}")
    print(f"[INFO] Magic guess: {kind}")

    if size < 32:
        raise ValueError("File is extremely small—likely an incomplete upload.")

    # Try ZIP first if it looks like/claims to be zip
    if kind == "zip" or archive_path.suffix.lower() == ".zip":
        try:
            with zipfile.ZipFile(archive_path, "r") as zf:
                target = out_dir / (archive_path.stem + "_extracted")
                target.mkdir(parents=True, exist_ok=True)
                zf.extractall(target)
            print(f"[OK] Extracted as ZIP -> {target.resolve()}")
            return target
        except zipfile.BadZipFile as e:
            print(f"[WARN] ZIP failed: {e}")

    # Try tar family
    for mode, label in [("r:gz", "tar.gz"), ("r:bz2", "tar.bz2"), ("r:xz", "tar.xz"), ("r:", "tar")]:
        try:
            with tarfile.open(archive_path, mode) as tf:
                target = out_dir / (archive_path.stem + "_extracted")
                target.mkdir(parents=True, exist_ok=True)
                tf.extractall(target)
            print(f"[OK] Extracted as {label} -> {target.resolve()}")
            return target
        except tarfile.ReadError:
            continue
        except Exception as e:
            print(f"[WARN] {label} attempt failed: {e}")

    # As a last resort, let shutil guess from extension (often just repeats the above)
    try:
        target = out_dir / (archive_path.stem + "_extracted")
        target.mkdir(parents=True, exist_ok=True)
        shutil.unpack_archive(str(archive_path), str(target))
        print(f"[OK] Extracted via shutil.unpack_archive -> {target.resolve()}")
        return target
    except Exception as e:
        pass

    # Helpful diagnostics
    if kind.startswith("html"):
        raise ValueError(
            "This file appears to be HTML (e.g., a sign-in/download page) saved with .zip. "
            "Re-download the dataset file itself."
        )

    raise ValueError(
        "Could not extract the file. It may be corrupted or not a supported archive format. "
        "If it’s supposed to be a .zip, re-upload/re-download it; otherwise provide the correct extension."
    )

# ===== Use it =====
archive = Path("miniDL_1350_bundle.zip")  # change name if needed
smart_extract(archive)

[INFO] File: miniDL_1350_bundle.zip | Size: 297.00 MB | MD5: e13a06a7fbbc4e111ca2018f1680ec37
[INFO] Magic guess: zip
[OK] Extracted as ZIP -> /largeDataVolume/zadid_workspace/miniDL_1350_bundle_extracted


PosixPath('miniDL_1350_bundle_extracted')

# Optuna w/ miniDL subset

In [ ]:
# ============================ QUIET LOGS ============================
import os, warnings, unicodedata, re
warnings.filterwarnings("ignore", message=r"Deterministic behavior.*")
warnings.filterwarnings("ignore", message=r".*does not have a deterministic implementation.*")
warnings.filterwarnings("ignore", category=UserWarning, module="torch")
os.environ.pop("CUBLAS_WORKSPACE_CONFIG", None)

# ============================ IMPORTS ===============================
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as T
from torch import optim

import optuna

# Your model
from iSyncTab import iSyncTab, set_seed

# ============================ USER PATHS ============================
CSV_PATH = "miniDL_1350_bundle_extracted/DL_info_processed.csv"   # <-- point to your CSV
IMG_ROOT = "miniDL_1350_bundle_extracted/images"                  # <-- folder containing PNGs in File_name

# ============================ DEVICE PICKER =========================
def pick_device(prefer=(6, 7)):
    if torch.cuda.is_available():
        for idx in prefer:
            if idx < torch.cuda.device_count():
                torch.cuda.set_device(idx)
                return torch.device(f"cuda:{idx}")
        # fallback to any visible CUDA
        torch.cuda.set_device(0)
        return torch.device("cuda:0")
    return torch.device("cpu")

set_seed(123)
device = pick_device()
print("Device:", device)

# ============================ HELPERS ===============================
def _slug(s: str) -> str:
    s = unicodedata.normalize("NFKD", str(s))
    s = s.encode("ascii", "ignore").decode("ascii")
    s = s.lower()
    return re.sub(r"[^a-z0-9]+", "", s)

def map_split_to_3class(v):
    """
    Map Train_Val_Test to 3 classes (0,1,2).
    Accepts numeric {1,2,3} or strings {'1','2','3'}.
    We'll interpret: 1->train, 2->val, 3->test
    """
    if pd.isna(v): return 1
    s = str(v).strip()
    if s in {"1", "1.0"}: return 0
    if s in {"2", "2.0"}: return 1
    if s in {"3", "3.0"}: return 2
    # if someone wrote 'train','val','test'
    sL = s.lower()
    if "train" in sL: return 0
    if "val" in sL:   return 1
    if "test" in sL:  return 2
    return 1

# ============================ DATASET ===============================
class DeepLesionSubset(Dataset):
    """
    Multimodal loader for the provided DeepLesion-style CSV.
      - Image path: IMG_ROOT / File_name
      - Tabular features:
         * Numeric: all numeric columns except the target
         * Categorical: string-like columns (e.g., Patient_gender)
      - Target: Train_Val_Test mapped to 3 classes (0=train, 1=val, 2=test)
    """
    def __init__(self, csv_path, img_root, transform=None):
        self.df = pd.read_csv(csv_path)
        self.img_root = Path(img_root)
        self.transform = transform

        # ----- required columns -----
        assert "File_name" in self.df.columns, "CSV must contain 'File_name' column."
        assert "Train_Val_Test" in self.df.columns, "CSV must contain 'Train_Val_Test' column."

        # ----- resolve image paths & keep valid rows -----
        kept, paths = [], []
        for i, row in self.df.iterrows():
            p = self.img_root / str(row["File_name"])
            if p.exists():
                kept.append(i); paths.append(p)
        if not kept:
            raise RuntimeError("No valid images found. Check CSV 'File_name' and IMG_ROOT.")
        self.df = self.df.iloc[kept].reset_index(drop=True)
        self.img_paths = paths

        # ----- target -----
        self.classes = ["train", "val", "test"]
        self.y = torch.tensor([map_split_to_3class(v) for v in self.df["Train_Val_Test"]], dtype=torch.long)

        # ----- build feature lists -----
        # Exclude these from tabular features
        exclude = {"File_name", "Train_Val_Test"}
        # Separate numeric vs categorical by dtype
        num_cols = []
        cat_cols = []
        for col in self.df.columns:
            if col in exclude:
                continue
            # treat strings/objects as categorical
            if pd.api.types.is_numeric_dtype(self.df[col]):
                num_cols.append(col)
            else:
                cat_cols.append(col)

        # (Optional) if you want to force some IDs to categorical instead of numeric:
        # ids_as_cat = {"Patient_gender"}
        # cat_cols = sorted(set(cat_cols) | (ids_as_cat & set(self.df.columns)))
        # num_cols = [c for c in num_cols if c not in ids_as_cat]

        self.num_cols = num_cols
        self.cat_cols = cat_cols
        self.text_cols = []  # none here

        # ----- numeric tensor -----
        if self.num_cols:
            num_df = self.df[self.num_cols].astype(float).fillna(0.0)
            self.x_num = torch.tensor(num_df.to_numpy(copy=True), dtype=torch.float32)
        else:
            self.x_num = torch.zeros((len(self.df), 0), dtype=torch.float32)

        # ----- categorical -> ids (per-column vocab, -1 for missing/unseen) -----
        self.x_cat = None
        self._cat_vocabs = {}
        if self.cat_cols:
            cat_arrays = []
            for col in self.cat_cols:
                vals = self.df[col].astype("object")
                uniq = sorted({str(v) for v in vals.dropna().unique().tolist()})
                vocab = {tok: i for i, tok in enumerate(uniq)}
                self._cat_vocabs[col] = vocab
                ids = [vocab.get(str(v), -1) if pd.notna(v) else -1 for v in vals]
                cat_arrays.append(torch.tensor(ids, dtype=torch.long))
            self.x_cat = torch.stack(cat_arrays, dim=1)

        # ----- dummy text channel so EmbeddingBag never sees O_text=0 -----
        self.x_text = torch.zeros((len(self.df), 1, 1), dtype=torch.long)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        # Image (convert to RGB for ImageNet normalization)
        img = Image.open(self.img_paths[idx])
        if img.mode != "RGB":
            img = img.convert("RGB")
        if self.transform:
            img = self.transform(img)

        x_num  = self.x_num[idx] if self.x_num.numel() else torch.zeros(0, dtype=torch.float32)
        x_cat  = self.x_cat[idx] if self.x_cat is not None else torch.zeros(0, dtype=torch.long)
        x_text = self.x_text[idx]  # (1,1)
        y      = self.y[idx]
        x_tab = {"num": x_num.unsqueeze(0), "cat": x_cat.unsqueeze(0), "text": x_text.unsqueeze(0)}
        return x_tab, img, y

# -------------- Collate --------------
def collate(batch):
    x_tab_b, x_img_b, y_b = zip(*batch)
    x_num_b  = torch.cat([b["num"]  for b in x_tab_b], dim=0)
    x_cat_b  = torch.cat([b["cat"]  for b in x_tab_b], dim=0) if x_tab_b[0]["cat"].numel() else torch.zeros((len(x_tab_b),0), dtype=torch.long)
    x_text_b = torch.cat([b["text"] for b in x_tab_b], dim=0)
    x_tab_batch = {"num": x_num_b, "cat": x_cat_b, "text": x_text_b}
    x_img_b = torch.stack(x_img_b)
    y_b     = torch.tensor(y_b)
    return x_tab_batch, x_img_b, y_b

def accuracy_from_logits(logits, y):
    return (logits.argmax(dim=1) == y).float().mean().item()

# ========================== DATA & LOADERS ===========================
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])  # ImageNet
])

ds = DeepLesionSubset(CSV_PATH, IMG_ROOT, transform=transform)

n_classes = 3  # (train, val, test)
N = len(ds)
n_train = int(round(0.64 * N))
n_val   = int(round(0.16 * N))
n_test  = N - n_train - n_val
assert n_train>0 and n_val>0 and n_test>0, f"Bad split sizes (N={N})."

g = torch.Generator().manual_seed(123)
ds_train, ds_val, ds_test = random_split(ds, [n_train, n_val, n_test], generator=g)

def make_loaders(batch_size):
    pin = device.type == "cuda"
    dl_train = DataLoader(ds_train, batch_size=batch_size, shuffle=True,
                          num_workers=0, pin_memory=pin, collate_fn=collate)
    dl_val   = DataLoader(ds_val,   batch_size=batch_size, shuffle=False,
                          num_workers=0, pin_memory=pin, collate_fn=collate)
    dl_test  = DataLoader(ds_test,  batch_size=batch_size, shuffle=False,
                          num_workers=0, pin_memory=pin, collate_fn=collate)
    return dl_train, dl_val, dl_test

# token hint = numeric + categorical + (dummy) text(=1)
NUM_TAB_FEATURES = len(ds.num_cols) + len(ds.cat_cols) + 1

# ============================ MODEL ================================
def build_model(params):
    core = dict(
        num_tab_features = NUM_TAB_FEATURES,
        num_classes      = n_classes,
        num_clusters     = params["num_clusters"],
        metric           = params["metric"],
        lambda_fs        = params["lambda_fs"],
        pretrained_resnet= True
    )
    opt_kwargs = dict(
        d_model           = params.get("d_model"),
        linformer_depth   = params.get("linformer_depth"),
        linformer_heads   = params.get("linformer_heads"),
        linformer_k       = params.get("linformer_k"),
        num_memory_tokens = params.get("num_memory_tokens"),
    )
    opt_kwargs = {k:v for k,v in opt_kwargs.items() if v is not None}
    try:
        model = iSyncTab(**core, **opt_kwargs).to(device)
    except TypeError:
        model = iSyncTab(**core).to(device)
    return model

def run_epoch(model, loader, opt=None):
    train = opt is not None
    model.train(train)
    losses, accs = [], []
    for x_tab, img, y in loader:
        img, y = img.to(device), y.to(device)
        x_tab = {k: v.to(device) for k,v in x_tab.items()}
        out = model(x_tab, img, y=y)
        if train:
            opt.zero_grad()
            out["loss"].backward()
            opt.step()
        losses.append(out["loss"].item())
        accs.append(accuracy_from_logits(out["logits"], y))
    return (float(np.mean(losses)) if losses else 0.0,
            float(np.mean(accs))   if accs   else 0.0)

# ================================ OPTUNA =============================
PENALIZE_LAMBDA = 0.02
EPOCHS_TUNE     = 3
FINAL_EPOCHS    = 25

def suggest_params(trial):
    return {
        "d_model": trial.suggest_categorical("d_model", [128, 192, 256]),
        "linformer_heads": trial.suggest_categorical("linformer_heads", [2, 4, 8]),
        "linformer_depth": trial.suggest_int("linformer_depth", 3, 5),
        "linformer_k": trial.suggest_categorical("linformer_k", [16, 32, 64]),
        "num_memory_tokens": trial.suggest_int("num_memory_tokens", 1, 3),
        "num_clusters": trial.suggest_int("num_clusters", 3, 7),
        "metric": trial.suggest_categorical("metric", ["variance", "euclidean", "cosine", "correlation", "kl", "js", "manhattan"]),
        "lr": trial.suggest_float("lr", 1e-4, 4e-4, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 2e-4, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [8, 16, 32]),
        "lambda_fs": trial.suggest_float("lambda_fs", 0.0, 0.3),
    }

def objective(trial):
    params = suggest_params(trial)
    if params["d_model"] % params["linformer_heads"] != 0:
        raise optuna.TrialPruned()

    set_seed(1000 + trial.number)
    dl_train, dl_val, _ = make_loaders(params["batch_size"])

    model = build_model(params)
    opti  = optim.AdamW(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])

    # warmup (optional)
    try:
        x_tab_b, img_b, y_b = next(iter(dl_train))
        img_b, y_b = img_b.to(device), y_b.to(device)
        x_tab_b = {k: v.to(device) for k, v in x_tab_b.items()}
        _ = model(x_tab_b, img_b, y=y_b)
    except StopIteration:
        pass

    for _ in range(EPOCHS_TUNE):
        run_epoch(model, dl_train, opt=opti)

    _, val_acc = run_epoch(model, dl_val, opt=None)
    return float(val_acc - PENALIZE_LAMBDA * params["lambda_fs"])

# Run study
N_TRIALS = 20
study = optuna.create_study(direction="maximize", study_name="iSyncTab_DL",
                            sampler=optuna.samplers.TPESampler(seed=123))
study.optimize(objective, n_trials=N_TRIALS, gc_after_trial=True, show_progress_bar=False)

print("\n=== Optuna best (validation objective) ===")
print(f"Best score = {study.best_value:.4f}")
print("Best params:")
for k, v in study.best_trial.params.items():
    print(f"  {k}: {v}")

best = study.best_trial.params

# ==================== RETRAIN BEST + TEST EVAL =======================
set_seed(777)
dl_train_best, dl_val_best, dl_test_best = make_loaders(best["batch_size"])

# Merge train+val
trainval_indices = list(range(len(ds_train))) + [len(ds_train) + i for i in range(len(ds_val))]
subset = torch.utils.data.Subset(ds, trainval_indices)
dl_trainval = DataLoader(subset, batch_size=best["batch_size"], shuffle=True,
                         num_workers=0, pin_memory=(device.type=="cuda"), collate_fn=collate)

model_best = build_model(best)
opt_best   = optim.AdamW(model_best.parameters(), lr=best["lr"], weight_decay=best["weight_decay"])

# warmup once
try:
    x_tab_b, img_b, y_b = next(iter(dl_trainval))
    img_b, y_b = img_b.to(device), y_b.to(device)
    x_tab_b = {k: v.to(device) for k, v in x_tab_b.items()}
    _ = model_best(x_tab_b, img_b, y=y_b)
except StopIteration:
    pass

for _ in range(FINAL_EPOCHS):
    run_epoch(model_best, dl_trainval, opt=opt_best)

test_loss, test_acc = run_epoch(model_best, dl_test_best, opt=None)
print("\n=== FINAL TEST RESULTS ===")
print(f"Test accuracy: {test_acc:.4f}")
print(f"Test loss:     {test_loss:.4f}")
print("\n=== Best config used for test ===")
for k, v in best.items():
    print(f"  {k}: {v}")

Device: cuda:6


[I 2025-11-10 23:22:35,955] A new study created in memory with name: iSyncTab_DL
[I 2025-11-10 23:36:27,455] Trial 0 finished with value: 0.8243690434082824 and parameters: {'d_model': 128, 'linformer_heads': 4, 'linformer_depth': 5, 'linformer_k': 16, 'num_memory_tokens': 2, 'num_clusters': 6, 'metric': 'correlation', 'lr': 0.0002090220546562389, 'weight_decay': 2.882541940368984e-05, 'batch_size': 8, 'lambda_fs': 0.21673301477106646}. Best is trial 0 with value: 0.8243690434082824.
[I 2025-11-10 23:49:36,752] Trial 1 finished with value: 0.8400726328454222 and parameters: {'d_model': 192, 'linformer_heads': 4, 'linformer_depth': 4, 'linformer_k': 32, 'num_memory_tokens': 1, 'num_clusters': 5, 'metric': 'euclidean', 'lr': 0.0003323304105591294, 'weight_decay': 3.769687142814138e-06, 'batch_size': 16, 'lambda_fs': 0.1838683577288903}. Best is trial 1 with value: 0.8400726328454222.
[I 2025-11-11 00:02:45,296] Trial 2 finished with value: 0.8286069284635336 and parameters: {'d_model': 1


=== Optuna best (validation objective) ===
Best score = 0.8480
Best params:
  d_model: 192
  linformer_heads: 8
  linformer_depth: 5
  linformer_k: 32
  num_memory_tokens: 2
  num_clusters: 4
  metric: manhattan
  lr: 0.00016360038864323952
  weight_decay: 7.088559782079231e-06
  batch_size: 32
  lambda_fs: 0.05391807890933545

=== FINAL TEST RESULTS ===
Test accuracy: 0.8563
Test loss:     0.2129

=== Best config used for test ===
  d_model: 192
  linformer_heads: 8
  linformer_depth: 5
  linformer_k: 32
  num_memory_tokens: 2
  num_clusters: 4
  metric: manhattan
  lr: 0.00016360038864323952
  weight_decay: 7.088559782079231e-06
  batch_size: 32
  lambda_fs: 0.05391807890933545
